<h1>Introduction to Mediaflux</h1>

https://www.arcitecta.com/mediaflux/features/

<b>What is Mediaflux?</b>

Mediaflux is the framework used for storing data on Pawsey.

Key concepts:

<b>assets</b> - data and metadata<br/>
<b>namespaces</b> - remote folder on the mediaflux server

</pre>

- Metadata is XML-based 
- Mediaflux has it's own query language 

The main commands, interacting with assets and associated metadata are:
<pre>
asset.get - display metadata for an asset
asset.set - alter metadata for an asset
asset.query - find assets based on metadata queries
</pre>

<h2>Mediaflux commands can be executed through pshell</h2><br/>
<b>Examples</b><br/><br/>
asset.query :where "namespace>='/projects/WACopernicus/Sentinel-2/MSI/L1C/2021/2021-07' and name='*20210721*50HLG*.zip'" 
:action get-path<br/>

asset.query :where "namespace>='/projects/WACopernicus/Sentinel-2/MSI/L1C/2021/2021-07'and name='*20210702*.zip'" :action get-path<br/>

asset.get :id -only-if-exists true "path=/projects/WACopernicus/temp/test.txt"  :xpath -ename id id :xpath -ename<br/>
<h2>Mediaflux commands can also be executed through Python</h2><br/>

<b>Example - opening a connection to the Mediaflux server</b>

Pawsey provides a Python wrapper over the Mediaflux API to simplify coding
<pre>
import mfclient
import os

try:
    mf_client = mfclient.mf_client('https', '443', 'data.pawsey.org.au')
    mf_client.login(token=PAWSEY_TOKEN)
except Exception as e:
    sys.stderr.write("Exception: {}".format(str(e)))
    sys.exit(1)
</pre>
<br/>
<b>Example  - Finding and downloding data</b>
<br/>

<pre>

PAWSEY_TOKEN = "token string goes here"
SRC_DATA_BASEPATH = "/projects/WACopernicus/Sentinel-2/MSI/L1C" # Sentinel-2 L1C collection
yearstr = '2021'  # Test date
yyyymmdd = '20211023'
mmstr = '10'
tile = '50HMK'  # Tile to search for

src_data_full_path = os.path.join(SRC_DATA_BASEPATH, yearstr, yearstr + "-" + mmstr)
query_command = "asset.query :where \"namespace>='" + src_data_full_path + "' and name=" + "'*" + \
       yyyymmdd + "*" + tile + "*.zip'" + "\"" + " :action get-path"

reply = mf_client.aterm_run(query_command) # Run search to find data

n = reply[0][0][0][0]
if n is None:
     sys.stderr.write("Error when querying\n")
     sys.exit(1)

p = n.find("path") # Get its location
if p is None:
     sys.stdout.write("No data found for {}, {}\n".format(tile, yyyymmdd))
     sys.exit(1)

file_name = p.text # Extract filename
</pre>
<br/>
<b>Download data</b>
<br/>

<pre>
reply = mf_client.aterm_run('asset.get :id -only-if-exists true "path=%s" :xpath content/size' % file_name
</pre>

- Returns json formatted reply to reflect outcome of request.
- If successful, data is downloaded to the current folder.
- If not, inspect contents of response to determine cause of error.